## Task 2: Parser Implementation

In this task, a rule-based parser is implemented to extract structured information from raw Telegram messages.  
The parser converts unstructured text into a tabular format suitable for analysis and modeling.


In [40]:
import json
import pandas as pd

In [41]:
# Load Raw Data:

with open("../data/raw/messages.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

df = pd.DataFrame(raw_data["messages"])
df.shape

(4596, 25)

In [42]:
# Import Parser:

import sys
import os

# Add project root to Python path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

from src.parsing import normalize_text, parse_message

In [43]:
# Apply Parser:

df_parsed = df.copy()

df_parsed["raw_text"] = df_parsed["text"].apply(normalize_text)
df_parsed["parsed"] = df_parsed["raw_text"].apply(parse_message)

df_parsed["parsed"].sample(3, random_state=42).tolist()

[{'professor_id': None,
  'professor_name_raw': 'سیاوش کاظمی راد',
  'department': 'مکانیک',
  'course_name': 'مدلسازی حرکات بدن',
  'rating_1': 8,
  'rating_2': 8,
  'rating_3': 8,
  'rating_4': 8,
  'rating_5': 10,
  'rating_6': 10,
  'grading_status_raw': 'دست باز و با ارفاق',
  'attendance_status_raw': 'حضور و غیاب نمی کند',
  'comment_text': 'دکتر کاظمی راد به شدت استاد با اخلاق و خوبی هستند. این درسشون با اینکه ترم اولی بود که ارائه می شد خیلی خوب بود و به شدت بازده خوبی داشت. درس پروژه محور هستش و بیشتر نمره رو پروژه ها تشکیل میده. پروژه ها شامل کدنویسی تو متلب (آنالیز گیت)، مدلسازی در OpenSim و بهینه سازی در متلب بود. به شدت به دوستانی که به حرکت شناسی بدن علاقه مند هستند توصیه می کنم.\n~~~~~~~~~~~~~~~~~\nبرای ثبت معرفی استاد به ربات زیر پیام بدید\n@ostad_elmosiBot\n\nکانال معرفی اساتید دانشگاه علم و صنعت\n@ostad_elmosi @ostad_elmosi',
  'term': 'بهمن 1400',
  'parse_error': False},
 {'professor_id': None,
  'professor_name_raw': 'حسینعلی پور',
  'department': 'مکانیک',
  'cour

In [44]:
# Unit Test: 

parse_message("🧑‍🏫 تست استاد\n📒 تست درس\nپیوستگی و یکپارچگی تدریس: 10")
parse_message("سلام")
parse_message("🧑‍🏫 تست استاد")

{'professor_id': None,
 'professor_name_raw': None,
 'department': None,
 'course_name': None,
 'rating_1': None,
 'rating_2': None,
 'rating_3': None,
 'rating_4': None,
 'rating_5': None,
 'rating_6': None,
 'grading_status_raw': None,
 'attendance_status_raw': None,
 'comment_text': None,
 'term': None,
 'parse_error': True}

In [45]:
parsed_df = pd.json_normalize(df_parsed["parsed"])
parsed_df.head()

,professor_id,professor_name_raw,department,course_name,rating_1,rating_2,rating_3,rating_4,rating_5,rating_6,grading_status_raw,attendance_status_raw,comment_text,term,parse_error
0,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None,True
1,None,None,خبر_مهم,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None,True
2,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None,True
3,None,سید محمد شهرتاش,برق,بررسی ۱-بررسی۲-حفاظت,8.0,9.0,7.0,9.0,8.0,10.0,منصفانه و هرچی خودت بگیری,حضور و غیاب نمی کند,چیزی اضافه ایی نیست\n~~~~~~~~~~~~~~~~~\nبرای ث...,مهر 99,False
4,None,دکتر هاجر فلاحتی,مهندسی_کامپیوتر,مدار منطقی- طراحی سیستم های کامپیوتری,1.0,2.0,1.0,3.0,2.0,10.0,نمره خوبی نمیشه ازشون گرفت,حضور مهم نیست اما تاثیر مثبت دارد,خوب درس نمیده اصلا، نمره ها رو خوب نمیده و اصل...,بهمن 98,False


In [52]:
# Build final DataFrame:

final_df = pd.concat(
    [
        df_parsed[["id", "date", "date_unixtime"]],
        parsed_df
    ],
    axis=1
)

final_df.shape

(4596, 18)

In [53]:
# Save parsed data (including parse_error flags)
final_df.to_csv(
    "../data/processed/parsed_messages.csv",
    index=False
)

### Parsed Dataset Summary

The parsing process successfully transformed raw Telegram messages into a structured tabular dataset.  
Messages that could not be reliably parsed were explicitly flagged and excluded from the clean dataset.  
This structured output serves as the foundation for subsequent statistical analysis and modeling.